# Notebook 16 — Nonlinear ensembles and a custom PyTorch neural network

**Development-only comparison.** Compare fixed-config Random Forest, Extra Trees, histogram gradient boosting, and a neural network implemented and trained in PyTorch with our frozen Logistic L2 benchmark and prior-class baseline. The latest 20% holdout is **not reused** and the prospective ledger is **not touched**.

The neural network uses explicit PyTorch layers, AdamW, minibatch optimization, cross-entropy loss, and chronological early stopping; no off-the-shelf neural estimator.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
if not (ROOT/'src'/'stock_ml_dynamic').exists(): raise RuntimeError('Open from the stock_ml_dynamic repository')
sys.path.insert(0,str(ROOT/'src'))
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.nonlinear_benchmark import run_nonlinear_benchmark
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd

required=('master_labeled.parquet','latest_inference.parquet','transformers.joblib','manifest.json')
candidates=[ROOT/'artifacts'/'master_tables'/'1d',ROOT/'notebooks'/'artifacts'/'master_tables'/'1d']
master_dir=next((p for p in candidates if all((p/f).exists() for f in required)),None)
if master_dir is None:raise FileNotFoundError('Run Notebook 03 first to build master artifacts')
master,latest,bundle,manifest=load_master(master_dir)
print('Master rows:',len(master),'symbols:',master.symbol.nunique())

## Fixed experimental settings

Start with `N_FOLDS=2` and `NEURAL_EPOCHS=8` for a smoke test, then run the default five folds. Ensembles may take several minutes per fold. GPU is used automatically for the neural network if PyTorch detects CUDA. All configurations are fixed before inspecting fold results; this is exploratory development CV, not a new final holdout.

In [ ]:
N_FOLDS=5
MAX_TRAIN_ROWS=60_000
TREE_TRAIN_ROWS=45_000
INCLUDE_NEURAL=True
NEURAL_EPOCHS=35
OUTPUT=ROOT/'artifacts'/'nonlinear_benchmark'/'1d'

results,summary,eligibility,development_end=run_nonlinear_benchmark(
    master,n_folds=N_FOLDS,max_train_rows=MAX_TRAIN_ROWS,tree_train_rows=TREE_TRAIN_ROWS,
    include_neural=INCLUDE_NEURAL,neural_epochs=NEURAL_EPOCHS,output_dir=OUTPUT)
print('Development boundary:',development_end)
print('Final holdout: NOT REUSED; prospective ledger: NOT TOUCHED')

In [ ]:
display(summary)
display(results[['fold','model','accuracy','balanced_accuracy','roc_auc','log_loss','fit_seconds']])

In [ ]:
fig,ax=plt.subplots(figsize=(11,5))
for name,g in results.groupby('model'):
    ax.plot(g.fold,g.roc_auc,marker='o',label=name)
ax.set(xlabel='Expanding chronological fold',ylabel='Macro OVR ROC-AUC',title='Development-only nonlinear benchmark')
ax.legend(loc='best',fontsize=8)
fig.tight_layout()
plt.show()
print('Saved:',OUTPUT)

## Interpretation

Inspect **ROC-AUC, log-loss, balanced accuracy, fold stability, and compute time**, not just accuracy. A model that beats Logistic L2 on a single fold is not necessarily better. The already-evaluated final holdout (37.83% accuracy, ROC-AUC 0.5604) is historical context only, **not a selection criterion**. Do not deploy or append experimental-model predictions to the frozen prospective ledger.

**Neural validation caveat:** the inner chronological slice controls early stopping; preprocessing is fitted across the entire outer-training window, so inner loss is not an unbiased model-selection score. The outer chronological test fold remains strictly out of sample.